# Cuestionario ampliado del Censo 2020 → tasas municipales con precisión

Hitos **T1–T2** de `docs/planes/04-medi-ampliado.md`. Descarga los microdatos del
Cuestionario ampliado (tabla `Viviendas_CA`, 32 estados), recodifica los
tres componentes del MEDI que faltaban y los estima **con factor de expansión
y error muestral** por municipio, estado y localidad de 50 000+ habitantes:

| indicador | definición | pregunta / variable |
|---|---|---|
| `p_lena` | % viviendas que cocinan con leña, carbón u *otro* combustible (≠ gas, electricidad), sobre las que cocinan | 8, `COMBUSTIBLE` (1 leña/carbón, 2 gas, 3 electricidad, 4 otro, 5 no cocinan, 9 n. e.) |
| `p_fogon_sin_chim` | % viviendas que cocinan con leña/carbón **sin** tubo o chimenea, sobre las que cocinan | 9, `ESTUFA` (1 con chimenea, 3 sin) |
| `p_sin_ac` | % viviendas sin aire acondicionado | 15, `AIRE_ACON` (5 sí, 6 no) |

Extras conservados (no MEDI): sin electricidad, focos ahorradores, boiler,
calentador solar, panel solar.

**Precisión.** Diseño estratificado por conglomerados: `ESTRATO`
(`ENT-MUN-LOC50K-nn`, es decir, estratos anidados en municipio y en
localidad de 50 k) y `UPM` (numerada dentro de cada estrato). Para una
proporción `p = Σwy / Σw` se usa linealización de Taylor + conglomerado
último: `z_i = w_i (y_i − p) / Σw`, se suma por UPM y la varianza es
`Σ_h n_h/(n_h−1) Σ_c (z_hc − z̄_h)²`. Estratos con una sola UPM aportan 0
(se cuentan). El CV se publica según la regla INEGI (ADR-0006): ≤ 15 % ok,
15–30 % con aviso, > 30 % poco preciso.

**Verificación** contra cifras oficiales: leña nacional 12.5 %; Chiapas
49.3, Oaxaca 46.1, Guerrero 40.8; los 32 estados contra
`uso_combustible.xlsx` y `cocina_c_chimenea.xlsx` (misma fuente); panel solar 0.8 %.

In [1]:
import hashlib
import io
import re
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw" / "INEGI" / "censo_ampliado_2020"
TAB = ROOT / "data" / "raw" / "INEGI" / "tabulados"
OUT_DIR = ROOT / "data" / "derived" / "INEGI" / "2020"
RAW.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://www.inegi.org.mx/contenidos/programas/ccpv/2020/microdatos/Censo2020_CA_{abr}_csv.zip"
# Abreviaturas que usa INEGI en los nombres de archivo (verificadas por HEAD el 2026-09-02).
ABR = {
    "01": "ags", "02": "bc", "03": "bcs", "04": "cam", "05": "coa", "06": "col", "07": "chs", "08": "chh",
    "09": "cdmx", "10": "dgo", "11": "gto", "12": "gro", "13": "hgo", "14": "jal", "15": "mex", "16": "mich",
    "17": "mor", "18": "nay", "19": "nl", "20": "oax", "21": "pue", "22": "qro", "23": "qroo", "24": "slp",
    "25": "sin", "26": "son", "27": "tab", "28": "tam", "29": "tla", "30": "ver", "31": "yuc", "32": "zac",
}
COLS = ["ENT", "MUN", "LOC50K", "ESTRATO", "UPM", "FACTOR", "CLAVIVP", "COMBUSTIBLE", "ESTUFA",
        "ELECTRICIDAD", "FOCOS", "FOCOS_AHORRA", "BOILER", "CALENTADOR_SOLAR", "AIRE_ACON", "PANEL_SOLAR", "TAMLOC"]
CV_OK, CV_AVISO = 15.0, 30.0

## 1. Descarga (idempotente, por tamaño remoto) y procedencia

In [2]:
def remote_size(url: str) -> int:
    with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60) as r:
        return int(r.headers["Content-Length"])


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(1 << 20):
            h.update(b)
    return h.hexdigest()


registro = []
for ee, abr in ABR.items():
    url, dest = URL.format(abr=abr), RAW / f"Censo2020_CA_{abr}_csv.zip"
    size = remote_size(url)
    assert size > 100_000, f"{ee} {abr}: la URL no devuelve un zip ({size} bytes)"
    if not (dest.exists() and dest.stat().st_size == size):
        with urllib.request.urlopen(url, timeout=600) as r, open(dest, "wb") as f:
            while b := r.read(1 << 20):
                f.write(b)
    registro.append((ee, abr, dest.name, dest.stat().st_size))
    print(f"{ee} {abr:4s} {dest.stat().st_size/1e6:6.1f} MB", end=" | " if int(ee) % 4 else "\n")
print(f"\ntotal {sum(r[3] for r in registro)/1e6:.0f} MB")

01 ags     3.2 MB | 02 bc      2.9 MB | 

03 bcs     1.7 MB | 04 cam     2.9 MB


05 coa     7.4 MB | 06 col     2.2 MB | 07 chs    34.8 MB | 

08 chh    10.2 MB
09 cdmx   10.3 MB | 10 dgo     8.1 MB | 

11 gto    13.7 MB | 12 gro    22.8 MB


...

21 pue    40.9 MB | 22 qro     4.9 MB | 23 qroo    2.7 MB | 

24 slp    11.8 MB
25 sin     5.6 MB | 26 son     9.1 MB | 

27 tab     5.4 MB | 28 tam     8.6 MB
29 tla    11.5 MB | 

30 ver    46.1 MB | 31 yuc    16.1 MB | 

32 zac     9.6 MB

total 489 MB


## 2. Lectura de `Viviendas_CA` (sólo columnas necesarias) y recodificación

In [3]:
def read_viviendas(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as z:
        name = next(n for n in z.namelist() if re.match(r"(?i)^viviendas\d*\.csv$", n.split("/")[-1]))
        raw = z.read(name)
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")
    return pd.read_csv(io.StringIO(text), dtype=str, usecols=COLS)


partes = [read_viviendas(RAW / f"Censo2020_CA_{abr}_csv.zip") for abr in ABR.values()]
viv = pd.concat(partes, ignore_index=True)
viv["w"] = pd.to_numeric(viv["FACTOR"], errors="coerce")
viv["psu"] = viv["ESTRATO"] + "|" + viv["UPM"]
viv["cve_ent"] = viv["ENT"].str.zfill(2)
viv["cve_mun"] = viv["cve_ent"] + viv["MUN"].str.zfill(3)
viv["cve_loc50k"] = viv["cve_mun"] + viv["LOC50K"].str.zfill(4)
assert viv["w"].notna().all() and (viv["w"] > 0).all()
assert viv["ESTRATO"].str[:6].eq(viv["cve_ent"] + "-" + viv["MUN"].str.zfill(3)).all(), "estratos no anidados en municipio"
print(f"{len(viv):,} viviendas en muestra; suma de factores {viv['w'].sum():,.0f} (VPH nacional 2020 ≈ 35.2 M)")
print("municipios:", viv["cve_mun"].nunique(), "| localidades 50k:", (viv["LOC50K"] != "0000").sum() and viv.loc[viv["LOC50K"] != "0000", "cve_loc50k"].nunique())


def binario(col: str, uno: set[str], cero: set[str]) -> pd.Series:
    s = viv[col]
    return pd.Series(np.where(s.isin(uno), 1.0, np.where(s.isin(cero), 0.0, np.nan)), index=viv.index)


cocina = viv["COMBUSTIBLE"].isin({"1", "2", "3", "4"})          # universo: viviendas que cocinan con respuesta
viv["lena"] = binario("COMBUSTIBLE", {"1", "4"}, {"2", "3"})
viv["lena_all"] = binario("COMBUSTIBLE", {"1"}, {"2", "3", "4", "5"})   # sólo leña/carbón, sobre todas: para verificar tabulado
viv["fogon_sin_chim"] = np.where(cocina, np.where(viv["ESTUFA"] == "3", 1.0, 0.0), np.nan)
viv["sin_chim_entre_lena"] = np.where(viv["COMBUSTIBLE"] == "1", binario("ESTUFA", {"3"}, {"1"}), np.nan)
viv["sin_ac"] = binario("AIRE_ACON", {"6"}, {"5"})
viv["sin_elec_ca"] = binario("ELECTRICIDAD", {"3"}, {"1"})
viv["boiler"] = binario("BOILER", {"1"}, {"2"})
viv["calentador_solar"] = binario("CALENTADOR_SOLAR", {"3"}, {"4"})
viv["panel_solar"] = binario("PANEL_SOLAR", {"7"}, {"8"})
focos = pd.to_numeric(viv["FOCOS"], errors="coerce")
ahorra = pd.to_numeric(viv["FOCOS_AHORRA"], errors="coerce")
viv["focos_ahorra_frac"] = np.where((focos > 0) & ahorra.notna(), ahorra / focos, np.nan)

INDICADORES = ["lena", "fogon_sin_chim", "sin_ac", "sin_elec_ca", "boiler", "calentador_solar", "panel_solar", "focos_ahorra_frac"]
for c in INDICADORES + ["lena_all", "sin_chim_entre_lena"]:
    print(f"  {c:20s} respuesta {viv[c].notna().mean():6.1%}  nacional ponderado {np.average(viv[c].dropna(), weights=viv.loc[viv[c].notna(), 'w'])*100:6.2f} %")

4,016,627 viviendas en muestra; suma de factores 34,987,915 (VPH nacional 2020 ≈ 35.2 M)
municipios: 2469 | localidades 50k: 232


  lena                 respuesta  97.8%  nacional ponderado  12.70 %
  fogon_sin_chim       respuesta  97.8%  nacional ponderado   9.77 %
  sin_ac               respuesta  99.7%  nacional ponderado  82.84 %
  sin_elec_ca          respuesta  99.7%  nacional ponderado   0.71 %
  boiler               respuesta  99.7%  nacional ponderado  48.07 %
  calentador_solar     respuesta  99.7%  nacional ponderado  10.34 %
  panel_solar          respuesta  99.7%  nacional ponderado   0.81 %
  focos_ahorra_frac    respuesta  97.8%  nacional ponderado  87.57 %
  lena_all             respuesta  99.7%  nacional ponderado  12.49 %
  sin_chim_entre_lena  respuesta  33.5%  nacional ponderado  77.17 %


## 3. Estimación por dominio con error muestral

In [4]:
def estimar(df: pd.DataFrame, dominio: str, ycol: str) -> pd.DataFrame:
    """Proporción ponderada por dominio + error estándar (Taylor, conglomerado último)."""
    d = df.loc[df[ycol].notna(), [dominio, "ESTRATO", "psu", "w", ycol]].copy()
    W = d.groupby(dominio)["w"].transform("sum")
    p = (d["w"] * d[ycol]).groupby(d[dominio]).transform("sum") / W
    d["z"] = d["w"] * (d[ycol] - p) / W
    z_psu = d.groupby([dominio, "ESTRATO", "psu"])["z"].sum().reset_index()
    g = z_psu.groupby([dominio, "ESTRATO"])["z"]
    n_h = g.transform("count")
    zbar = g.transform("mean")
    z_psu["contrib"] = np.where(n_h > 1, n_h / (n_h - 1) * (z_psu["z"] - zbar) ** 2, 0.0)
    var = z_psu.groupby(dominio)["contrib"].sum()
    out = pd.DataFrame({
        "p": (d["w"] * d[ycol]).groupby(d[dominio]).sum() / d.groupby(dominio)["w"].sum() * 100,
        "n": d.groupby(dominio).size(),
        "n_psu": d.groupby(dominio)["psu"].nunique(),
    })
    out["se"] = np.sqrt(var.reindex(out.index).fillna(0.0)) * 100
    out["cv"] = np.where(out["p"] > 0, out["se"] / out["p"] * 100, np.nan)
    return out


def tabla(dominio: str) -> pd.DataFrame:
    """Una fila por dominio presente en la muestra; nulo donde no hay respuestas válidas."""
    dominios = sorted(viv[dominio].unique())
    partes = {}
    for c in INDICADORES:
        e = estimar(viv, dominio, c).reindex(dominios)
        partes[f"p_{c}"] = e["p"]
        partes[f"cv_{c}"] = e["cv"]
        partes[f"n_{c}"] = e["n"].fillna(0).astype("int64")
    t = pd.DataFrame(partes, index=pd.Index(dominios, name=dominio))
    t["n_muestra"] = viv.groupby(dominio).size()
    t["n_psu"] = viv.groupby(dominio)["psu"].nunique()
    t["viv_expandidas"] = viv.groupby(dominio)["w"].sum().round().astype("int64")
    return t


ent = tabla("cve_ent")
mun = tabla("cve_mun")
loc = tabla("cve_loc50k")
loc = loc[~loc.index.str.endswith("0000")]           # sólo localidades de 50 k (LOC50K ≠ 0000)
print("estados", len(ent), "| municipios", len(mun), "| localidades 50k", len(loc))
assert len(mun) == 2469

# Municipios sin ninguna respuesta válida: en el ampliado vienen completos con
# COBERTURA = 3 (viviendas sin información de ocupantes, CLAVIVP = 99). Es un
# hueco de la fuente; quedan en nulo y se documenta.
sin_dato = mun.index[mun["p_sin_ac"].isna()].tolist()
cob = viv[viv["cve_mun"].isin(sin_dato)].groupby("cve_mun").agg(filas=("w", "size"), clavivp=("CLAVIVP", lambda s: ",".join(sorted(s.unique()))))
print("municipios sin dato en el ampliado:", sin_dato)
print(cob.to_string())
assert len(sin_dato) <= 5, sin_dato

estados 32 | municipios 2469 | localidades 50k 232
municipios sin dato en el ampliado: ['04012', '07125', '29048']
         filas clavivp
cve_mun               
04012      369      99
07125      244      99
29048       13      99


## 4. Verificación contra cifras oficiales

In [5]:
nac_lena_all = np.average(viv["lena_all"].dropna(), weights=viv.loc[viv["lena_all"].notna(), "w"]) * 100
nac_panel = np.average(viv["panel_solar"].dropna(), weights=viv.loc[viv["panel_solar"].notna(), "w"]) * 100
print(f"leña o carbón nacional (sobre todas las viviendas): {nac_lena_all:.2f} %  (boletín INEGI: 12.5 %)")
print(f"panel solar nacional: {nac_panel:.2f} %  (boletín: 0.8 %)")
assert abs(nac_lena_all - 12.5) < 0.15 and abs(nac_panel - 0.8) < 0.1

ent_lena_all = estimar(viv, "cve_ent", "lena_all")["p"]
comb = pd.read_excel(TAB / "uso_combustible.xlsx", header=None, skiprows=4, dtype=str).iloc[:, :4]
comb.columns = ["cve", "entidad", "vph", "lena_tab"]
comb["cve"] = comb["cve"].str.zfill(2)
comb["lena_tab"] = pd.to_numeric(comb["lena_tab"])
cmp = comb.set_index("cve")[["entidad", "lena_tab"]].join(ent_lena_all.rename("lena_micro"))
cmp["dif"] = (cmp["lena_micro"] - cmp["lena_tab"]).abs()
print("\nleña por estado, |microdatos − tabulado| máximo:", round(cmp["dif"].max(), 3))
print(cmp.sort_values("lena_tab", ascending=False).head(5).round(2).to_string())
print("estados con mayor diferencia (el tabulado reparte distinto los no especificados):")
print(cmp.sort_values("dif", ascending=False).head(3).round(2).to_string())
assert cmp["dif"].max() < 0.5
for cve, esperado in {"07": 49.3, "20": 46.1, "12": 40.8}.items():
    assert abs(cmp.loc[cve, "lena_micro"] - esperado) < 0.2, cve

chim = pd.read_excel(TAB / "cocina_c_chimenea.xlsx", header=None, skiprows=3, dtype=str).iloc[:, [0, 3, 4]]
chim.columns = ["cve", "p_disponen", "p_no_disponen"]
chim["cve"] = chim["cve"].str.zfill(2)
chim["p_no_disponen"] = pd.to_numeric(chim["p_no_disponen"])
ent_sinchim = estimar(viv, "cve_ent", "sin_chim_entre_lena")["p"]
cmp2 = chim.set_index("cve")[["p_no_disponen"]].join(ent_sinchim.rename("sin_chim_micro"))
# El tabulado reparte "no especificado" aparte; microdatos sólo tienen 1/3, así que
# la comparación válida es sin_chim_micro ≈ PND / (P_Disponen + PND).
chim["p_disponen"] = pd.to_numeric(chim["p_disponen"])
cmp2["esperado"] = (chim.set_index("cve")["p_no_disponen"] / (chim.set_index("cve")["p_disponen"] + chim.set_index("cve")["p_no_disponen"]) * 100)
cmp2["dif"] = (cmp2["sin_chim_micro"] - cmp2["esperado"]).abs()
print("\nsin chimenea entre quienes usan leña, |microdatos − tabulado| máximo:", round(cmp2["dif"].max(), 3))
print(cmp2.sort_values("dif", ascending=False).head(3).round(2).to_string())
assert cmp2["dif"].max() < 1.0

print("\nprecisión municipal (CV %, p_sin_ac):")
print(mun["cv_sin_ac"].describe(percentiles=[0.5, 0.75, 0.9]).round(1).to_string())
for c in ["lena", "fogon_sin_chim", "sin_ac"]:
    cv = mun[f"cv_{c}"]
    print(f"  {c:16s} CV≤15: {(cv <= CV_OK).mean():5.1%} | 15–30: {((cv > CV_OK) & (cv <= CV_AVISO)).mean():5.1%} | >30 o nulo: {((cv > CV_AVISO) | cv.isna()).mean():5.1%}")

leña o carbón nacional (sobre todas las viviendas): 12.49 %  (boletín INEGI: 12.5 %)
panel solar nacional: 0.81 %  (boletín: 0.8 %)



leña por estado, |microdatos − tabulado| máximo: 0.406
      entidad  lena_tab  lena_micro   dif
cve                                      
07    Chiapas     49.34       49.43  0.09
20     Oaxaca     46.08       46.10  0.02
12   Guerrero     40.84       40.85  0.01
31    Yucatán     27.49       27.50  0.02
27    Tabasco     26.61       26.61  0.00
estados con mayor diferencia (el tabulado reparte distinto los no especificados):
      entidad  lena_tab  lena_micro   dif
cve                                      
04   Campeche     24.37       24.77  0.41
29   Tlaxcala      7.34        7.44  0.10
07    Chiapas     49.34       49.43  0.09



sin chimenea entre quienes usan leña, |microdatos − tabulado| máximo: 0.0
     p_no_disponen  sin_chim_micro  esperado  dif
cve                                              
31           95.49           95.58     95.58  0.0
02           80.28           83.99     83.99  0.0
16           63.93           63.98     63.98  0.0

precisión municipal (CV %, p_sin_ac):
count    2466.0
mean        1.1
std         2.4
min         0.0
50%         0.3
75%         0.9
90%         3.5
max        39.3
  lena             CV≤15: 60.3% | 15–30: 29.0% | >30 o nulo: 10.7%
  fogon_sin_chim   CV≤15: 51.8% | 15–30: 29.9% | >30 o nulo: 18.3%
  sin_ac           CV≤15: 99.4% | 15–30:  0.4% | >30 o nulo:  0.2%


## 5. Escritura

In [6]:
def calidad(cv: pd.Series) -> pd.Series:
    return pd.Series(np.select([cv.isna(), cv <= CV_OK, cv <= CV_AVISO], ["sin_dato", "ok", "aviso"], "poco_preciso"), index=cv.index)


def finalizar(t: pd.DataFrame, nombre_clave: str) -> pd.DataFrame:
    t = t.copy()
    for c in ["lena", "fogon_sin_chim", "sin_ac"]:
        t[f"calidad_{c}"] = calidad(t[f"cv_{c}"])
    t.index.name = nombre_clave
    return t.reset_index()


ent_out = finalizar(ent, "cve_ent")
mun_out = finalizar(mun, "cvegeo")
loc_out = finalizar(loc, "cve_loc")
ent_out.to_parquet(OUT_DIR / "ampliado_ent_2020.parquet", index=False)
mun_out.to_parquet(OUT_DIR / "ampliado_mun_2020.parquet", index=False)
loc_out.to_parquet(OUT_DIR / "ampliado_loc50k_2020.parquet", index=False)
for n in ["ampliado_ent_2020", "ampliado_mun_2020", "ampliado_loc50k_2020"]:
    p = OUT_DIR / f"{n}.parquet"
    print("escrito", p.name, f"{p.stat().st_size/1e6:.2f} MB", pd.read_parquet(p).shape)

hoy = datetime.now(timezone.utc).strftime("%Y-%m-%d")
lineas = "\n".join(f"| {ee} | {abr} | `{name}` | {size:,} |" for ee, abr, name, size in registro)
(RAW / "SOURCE.md").write_text(f"""# Censo de Población y Vivienda 2020 — Cuestionario ampliado, microdatos (INEGI)

- **Producto:** Microdatos del Cuestionario ampliado (muestra probabilística, ~4 M viviendas), tabla `Viviendas_CA`.
- **Descarga:** `{URL}` (una por estado; abreviaturas de la tabla)
- **Fecha de descarga:** {hoy}
- **Representatividad (INEGI):** nacional, entidad, cada municipio/demarcación y localidades de 50 000+ habitantes; usar con estadísticas de precisión.
- **Licencia:** Términos de Libre Uso de la Información del INEGI.

| ent | abr | archivo | bytes |
|---|---|---|---|
{lineas}

Generado por `notebooks/008_AMPLIADO_mun.ipynb`.
""", encoding="utf-8")
print("SOURCE.md escrito")

escrito ampliado_ent_2020.parquet 0.03 MB (32, 31)
escrito ampliado_mun_2020.parquet 0.50 MB (2469, 31)
escrito ampliado_loc50k_2020.parquet 0.07 MB (232, 31)
SOURCE.md escrito
